### Метаклассы

In [1]:
class MyClass:
    pass

In [2]:
class MyClass(object):
    pass

In [3]:
print(MyClass.__bases__)

(<class 'object'>,)


In [4]:
s = "hello"
my_list = [1, 2]
class MyClass: pass

print(type(s))
print(type(my_list))
print(type(MyClass)) 

<class 'str'>
<class 'list'>
<class 'type'>


In [5]:
def my_method(self):
    print("Hello from dynamic class!")

# MyDynamicClass = type(имя_класса, кортеж_родителей, словарь_атрибутов)
MyDynamicClass = type("MyDynamicClass", (object,), {"x": 10, "hello": my_method})

dyn_obj = MyDynamicClass()
print(dyn_obj.x)
dyn_obj.hello()
print(type(dyn_obj))

10
Hello from dynamic class!
<class '__main__.MyDynamicClass'>


In [6]:
class UpperAttrMetaclass(type):
    def __new__(cls, name, bases, attrs):
        uppercase_attrs = {}
        for key, value in attrs.items():
            if not key.startswith('__'):
                uppercase_attrs[key.upper()] = value
            else:
                uppercase_attrs[key] = value
        
        # Важно: вызываем __new__ от родительского метакласса (type)
        return super().__new__(cls, name, bases, uppercase_attrs)

class MyModel(metaclass=UpperAttrMetaclass):
    some_attribute = "value"
    another_one = 123

model = MyModel()
# print(model.some_attribute)
print(model.SOME_ATTRIBUTE)
print(model.ANOTHER_ONE)

value
123


In [7]:
print(type(object))

<class 'type'>


In [8]:
print(type.__bases__)

(<class 'object'>,)


### Магические методы создания

In [ ]:
class ObjectSpy:
    """
    Класс, который логирует вызовы своих "магических" методов создания.
    """
    def __new__(cls, *args, **kwargs):
        # __new__ - это статический метод. Он получает класс (cls), а не экземпляр.
        print(f"--- 1. ВЫЗОВ __new__ ---")
        print(f"Получен класс: {cls.__name__}")
        
        # Запрашиваем создание пустого объекта у родителя (object)
        instance = super().__new__(cls)
        
        print(f"Создан пустой экземпляр: {instance} (id: {id(instance)})")
        # __new__ ОБЯЗАН вернуть экземпляр (или объект другого типа)
        return instance

    def __init__(self, name):
        # __init__ - это метод экземпляра. Он получает готовый объект как 'self'.
        print(f"--- 2. ВЫЗОВ __init__ ---")
        print(f"Получен экземпляр 'self': {self} (id: {id(self)})")
        print(f"Инициализируем его с атрибутом name='{name}'")
        self.name = name

print("Готовимся создать объект...")
spy = ObjectSpy("Агент 007")
print("\nСоздание завершено.")
print(f"Итоговый объект: {spy} (id: {id(spy)})")
print(f"Атрибут объекта: {spy.name}")

Готовимся создать объект...
--- 1. ВЫЗОВ __new__ ---
Получен класс: ObjectSpy
Создан пустой экземпляр: <__main__.ObjectSpy object at 0x108b28680> (id: 4440884864)
--- 2. ВЫЗОВ __init__ ---
Получен экземпляр 'self': <__main__.ObjectSpy object at 0x108b28680> (id: 4440884864)
Инициализируем его с атрибутом name='Агент 007'

Создание завершено.
Итоговый объект: <__main__.ObjectSpy object at 0x108b28680> (id: 4440884864)
Атрибут объекта: Агент 007


In [10]:
class ArgumentFlow:
    def __new__(cls, name, version):
        print(f"--- 1. __new__ получил: cls={cls.__name__}, name='{name}', version={version} ---")
        # Обычно __new__ не использует эти аргументы. Он просто создает объект.
        instance = super().__new__(cls)
        return instance

    def __init__(self, name, version):
        print(f"--- 2. __init__ получил: self={self}, name='{name}', version={version} ---")
        self.name = name
        self.version = version

print("Создаем объект с несколькими аргументами...")
flow = ArgumentFlow("Tracer", 2.0)
print(f"\nОбъект создан. Атрибуты: name='{flow.name}', version={flow.version}")

Создаем объект с несколькими аргументами...
--- 1. __new__ получил: cls=ArgumentFlow, name='Tracer', version=2.0 ---
--- 2. __init__ получил: self=<__main__.ArgumentFlow object at 0x108b28560>, name='Tracer', version=2.0 ---

Объект создан. Атрибуты: name='Tracer', version=2.0


In [ ]:
class Saboteur:
    def __new__(cls, action):
        print(f"--- __new__ в Saboteur. Действие: '{action}' ---")
        if action == "veto":
            print("Возвращаем None. __init__ не должен быть вызван.")
            return None
        elif action == "switch":
            print("Возвращаем строку. __init__ не должен быть вызван.")
            return "Я строка, а не объект Saboteur"
        else: # "normal"
            print("Возвращаем нормальный экземпляр.")
            return super().__new__(cls)

    def __init__(self, action):
        # Этот код выполнится только для "нормального" случая
        print(f"--- __init__ в Saboteur. Если вы это видите, __new__ вернул правильный объект. ---")
        self.action = action


print("--- Сценарий 1: Вето ---")
result1 = Saboteur("veto")
print(f"Результат: {result1}\n")

print("--- Сценарий 2: Подмена ---")
result2 = Saboteur("switch")
print(f"Результат: '{result2}', тип: {type(result2)}\n")

print("--- Сценарий 3: Нормальный ---")
result3 = Saboteur("normal")
print(f"Результат: {result3}")

In [ ]:
class SingletonLogger:
    _instance = None # Атрибут класса для хранения единственного экземпляра

    def __new__(cls):
        if cls._instance is None:
            print("Создаем единственный экземпляр логгера.")
            cls._instance = super().__new__(cls)
        else:
            print("Возвращаем уже существующий экземпляр.")
        return cls._instance
    
    # __init__ будет вызываться КАЖДЫЙ раз, что может быть нежелательно.
    # Поэтому часто его делают идемпотентным.
    def __init__(self):
        # Проверяем, был ли объект уже инициализирован
        if not hasattr(self, '_initialized'):
            print("Первичная инициализация логгера...")
            self.messages = []
            self._initialized = True

log1 = SingletonLogger()
log1.messages.append("Первое сообщение")

log2 = SingletonLogger() # __new__ вернет старый объект
log2.messages.append("Второе сообщение")

print(f"log1 это log2? -> {log1 is log2}")
print(f"Сообщения в log1: {log1.messages}")

In [11]:
class PositiveInteger(int):
    # Мы наследуемся от int, чтобы получить все его свойства
    def __new__(cls, value):
        if value <= 0:
            raise ValueError("Значение должно быть положительным")
        # Создаем и возвращаем экземпляр родительского класса (int)
        # с уже переданным значением
        return super().__new__(cls, value)
    
    # __init__ здесь не нужен, вся логика уже в __new__

# p_int = PositiveInteger(-5) # ValueError
p_int = PositiveInteger(10)
print(f"Значение: {p_int}, тип: {type(p_int)}")
print(f"Это подкласс int? {isinstance(p_int, int)}")
print(p_int + 5) # Работает как обычный int

Значение: 10, тип: <class '__main__.PositiveInteger'>
Это подкласс int? True
15


In [12]:
import dis

class SimpleObject:
    def __init__(self):
        self.value = 1

# Анализируем строку кода, которая создает экземпляр
print("--- Байт-код для 's = SimpleObject()' ---")
dis.dis("s = SimpleObject()")

--- Байт-код для 's = SimpleObject()' ---
  0           0 RESUME                   0

  1           2 PUSH_NULL
              4 LOAD_NAME                0 (SimpleObject)
              6 CALL                     0
             14 STORE_NAME               1 (s)
             16 RETURN_CONST             0 (None)


In [ ]:
a = 'Abc'
a.lower()


### \_\_del\_\_

In [ ]:
class LifeObserver:
    def __init__(self, name):
        self.name = name
        print(f"Объект {self.name} создан.")

    def __del__(self):
        print(f"!!! Объект {self.name} уничтожается. !!!")

# Демонстрация циклических ссылок
a = LifeObserver('A')
b = LifeObserver('B')

a.partner = b # a ссылается на b
b.partner = a # b ссылается на a

# Удаляем переменные, но не объекты!
del a
del b

# __del__ не вызывается! Объекты остались в памяти,
# т.к. счетчик ссылок каждого из них равен 1.
# Они будут удалены позже, сборщиком мусора для циклов,
# но на порядок вызовов __del__ нет никаких гарантий.
import gc
gc.collect() # Принудительно запускаем сборщик циклов

### \_\_dict\_\_ vs \_\_slots\_\_

In [13]:
class PersonDict:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p_dict = PersonDict("Иван", 30)
print(p_dict.__dict__)
p_dict.position = "developer" # Динамически добавляем атрибут
print(p_dict.__dict__)

{'name': 'Иван', 'age': 30}
{'name': 'Иван', 'age': 30, 'position': 'developer'}


In [15]:
class PersonSlots:
    __slots__ = ['name', 'age']

    def __init__(self, name, age):
        self.name = name
        self.age = age

p_slots = PersonSlots("Петр", 25)
print(p_slots.__dict__)
# p_slots.position = "developer" -> AttributeError: 'PersonSlots' object has no attribute 'position'

AttributeError: 'PersonSlots' object has no attribute '__dict__'

In [16]:
! pip install pympler

In [17]:
from pympler import asizeof

# Создаем много объектов
dict_list = [PersonDict(f"User{i}", i) for i in range(100_000)]
slots_list = [PersonSlots(f"User{i}", i) for i in range(100_000)]

size_dict = asizeof.asizeof(dict_list)
size_slots = asizeof.asizeof(slots_list)

print(f"Размер с __dict__: {size_dict / 1024 / 1024:.2f} MB")
print(f"Размер с __slots__: {size_slots / 1024 / 1024:.2f} MB")
print(f"Экономия: {(1 - size_slots / size_dict) * 100:.2f}%")

Размер с __dict__: 22.88 MB
Размер с __slots__: 13.73 MB
Экономия: 40.01%


### Callable

In [18]:
class Greeter:
    def __init__(self, greeting_word):
        print(f"--- Создан экземпляр Greeter с приветствием '{greeting_word}' ---")
        self.greeting = greeting_word # Сохраняем состояние

    def __call__(self, name):
        # Этот метод выполняется, когда мы "вызываем" экземпляр
        print(f"--- Вызов __call__ ---")
        return f"{self.greeting}, {name}!"

# Создаем два экземпляра с разным состоянием
english_greeter = Greeter("Hello")
spanish_greeter = Greeter("Hola")

print("\n--- Вызываем экземпляры как функции ---")
message1 = english_greeter("Alice")
print(message1)

message2 = spanish_greeter("Bob")
print(message2)

# Мы можем проверить, является ли объект вызываемым
print(f"\nЯвляется ли english_greeter вызываемым? {callable(english_greeter)}")
print(f"Является ли строка 'hello' вызываемой? {callable('hello')}")

--- Создан экземпляр Greeter с приветствием 'Hello' ---
--- Создан экземпляр Greeter с приветствием 'Hola' ---

--- Вызываем экземпляры как функции ---
--- Вызов __call__ ---
Hello, Alice!
--- Вызов __call__ ---
Hola, Bob!

Является ли english_greeter вызываемым? True
Является ли строка 'hello' вызываемой? False


In [ ]:
count_global = 0
def function_counter():
    global count_global
    count_global += 1
    return count_global

# Проблема: используется глобальная переменная,
# и мы не можем создать несколько независимых счетчиков.


class StatefulCounter:
    def __init__(self, start_value=0):
        self.current_value = start_value

    def __call__(self):
        result = self.current_value
        self.current_value += 1
        return result

print("--- Создаем два независимых счетчика ---")
counter_a = StatefulCounter(10) # Начнет с 10
counter_b = StatefulCounter(100) # Начнет со 100

print("Вызываем счетчик A:", counter_a())
print("Вызываем счетчик A:", counter_a())
print("Вызываем счетчик B:", counter_b())
print("Вызываем счетчик A:", counter_a())
print("Вызываем счетчик B:", counter_b())

### Bound

In [19]:
class MyClass:
    """Простой класс для демонстрации."""
    def my_method(self, x):
        # 'self' здесь - это тот экземпляр,
        # через который был вызван метод.
        print(f"Вызван my_method с self={self} и x={x}")

# Создаем экземпляр
instance = MyClass()
print(f"Наш экземпляр: {instance}\n")

# --- Получаем доступ к методу двумя способами ---

# 1. Через класс: получаем обычную функцию
unbound_function = MyClass.my_method
print(f"Доступ через класс: {unbound_function}")
print(f"   -> Тип: {type(unbound_function)}")

# 2. Через экземпляр: получаем "связанный метод"
bound_method = instance.my_method
print(f"Доступ через экземпляр: {bound_method}")
print(f"   -> Тип: {type(bound_method)}")

Наш экземпляр: <__main__.MyClass object at 0x10fbc2db0>

Доступ через класс: <function MyClass.my_method at 0x10aa40c20>
   -> Тип: <class 'function'>
Доступ через экземпляр: <bound method MyClass.my_method of <__main__.MyClass object at 0x10fbc2db0>>
   -> Тип: <class 'method'>


In [20]:
print("\n--- Заглянем внутрь связанного метода ---")

print(f"Атрибут __self__: {bound_method.__self__}")
print(f"   -> Это наш 'instance'? -> {bound_method.__self__ is instance}")

print(f"Атрибут __func__: {bound_method.__func__}")
print(f"   -> Это наша 'unbound_function'? -> {bound_method.__func__ is unbound_function}")


--- Заглянем внутрь связанного метода ---
Атрибут __self__: <__main__.MyClass object at 0x10fbc2db0>
   -> Это наш 'instance'? -> True
Атрибут __func__: <function MyClass.my_method at 0x10aa40c20>
   -> Это наша 'unbound_function'? -> True


In [21]:
print("\n--- Вызываем метод и симулируем вызов вручную ---")

print("Обычный вызов: bound_method(10)")
bound_method(10)

print("Ручная симуляция: bound_method.__func__(bound_method.__self__, 10)")
bound_method.__func__(bound_method.__self__, 10)


--- Вызываем метод и симулируем вызов вручную ---
Обычный вызов: bound_method(10)
Вызван my_method с self=<__main__.MyClass object at 0x10fbc2db0> и x=10
Ручная симуляция: bound_method.__func__(bound_method.__self__, 10)
Вызван my_method с self=<__main__.MyClass object at 0x10fbc2db0> и x=10
